# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.JOUETS = Set(initialize=[1, 2])
model.USINES = Set(initialize=[1, 2])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.USINES for j in model.JOUETS])

## 🔹 Parameters

In [ ]:
model.revenu = Param(model.JOUETS, initialize={1: 10.0, 2: 15.0}, within=NonNegativeReals)
model.cout_dem = Param(model.JOUETS, initialize={1: 50000.0, 2: 80000.0}, within=NonNegativeReals)
model.prod_max = Param(model.USINES, initialize={1: 500.0, 2: 700.0}, within=NonNegativeReals)
model.ponderation = Param(model.USINES, model.JOUETS, initialize={(1, 1): 0.02, (1, 2): 0.025, (2, 1): 0.025, (2, 2): 0.04}, within=NonNegativeReals)
model.bigM = Param(initialize=1000000.0, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.x = Var(model.JOUETS, domain=NonNegativeReals)
model.y = Var(model.JOUETS, domain=Binary)
model.z = Var(model.USINES, domain=Binary)

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=sum(model.z[u] for u in model.USINES) == 1)
model.c_for_0 = ConstraintList()
for u in model.USINES:
    model.c_for_0.add(sum(model.ponderation[u,j] * model.x[j] for j in model.JOUETS) <= model.prod_max[u] + model.bigM * ( 1 - model.z[u] ))
model.c_for_1 = ConstraintList()
for j in model.JOUETS:
    model.c_for_1.add(model.x[j] <= model.bigM * model.y[j])
# @BIN directive already handled in variable declarations

## 🔹 O

In [ ]:
# @BIN directive already handled in variable declarations

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.revenu[j] * model.x[j] - model.cout_dem[j] * model.y[j] for j in model.JOUETS), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('Solver status:', result.solver.status)
print('Termination condition:', result.solver.termination_condition)

## 📊 Valeurs optimales des variables

In [ ]:
for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v.name}')
    for index in v:
        print(f'   {index} = {v[index].value}')